# 🖐 Variantes de Control Gestual — Laboratorio

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Nomenclatura Oficial:** Procesamiento Digital de Imágenes
* **Nombre de Trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

Este laboratorio extiende el notebook **02_Control_Volumen_con_Manos** con tres variantes de control gestual usando **MediaPipe Hand Landmarker**. Cada variante implementa una técnica diferente para mapear el movimiento de la mano a una acción del sistema.

| Variante | Gesto | Control |
|---|---|---|
| **V1 — Pinch** | Distancia pulgar–índice | Volumen continuo |
| **V2 — Dos manos** | Altura de muñeca por mano | Volumen (derecha) + brillo (izquierda) |
| **V3 — Conteo de dedos** | Cantidad de dedos levantados | Volumen en 6 niveles discretos |

**Ejecutá cada sección de forma independiente.** Cerrá la ventana con **Q** antes de pasar a la siguiente.

## Objetivos

Al terminar este laboratorio vas a poder:

1. Implementar diferentes estrategias de mapeo entre gestos y valores de control.
2. Usar la distancia euclidiana entre landmarks como señal de entrada.
3. Diferenciar manos usando la etiqueta `handedness` de MediaPipe.
4. Detectar el estado de extensión de cada dedo comparando coordenadas de landmarks.

## Terminología clave (Microglosario)

* **Gesto pinch:** Acción de acercar o alejar los dedos pulgar e índice. *Como ajustar el volumen girando una perilla entre dos dedos: la distancia entre ellos es la magnitud de control.*
* **Distancia euclidiana normalizada:** Longitud de la línea recta entre dos landmarks en coordenadas [0,1], independiente de la resolución de la cámara. *Como medir en porcentaje del ancho de pantalla la separación entre dos puntos.*
* **Handedness (lateralidad):** Etiqueta que MediaPipe asigna a cada mano detectada ('Right' / 'Left'). *Como una pulsera de color en cada mano: permite distinguir cuál es cuál aunque ambas estén en el mismo cuadro.*
* **Brillo de imagen:** Factor de escala aplicado a los valores de cada píxel. *Como el botón de brillo de pantalla: multiplica la intensidad de cada píxel por un número.*
* **Dedo extendido:** Un dedo está extendido cuando la punta (*tip*) está más alejada de la palma que la articulación media (*PIP*). *En imagen: la punta tiene menor coordenada Y (está más arriba).*
* **Nivel discreto:** Valor que solo puede tomar un conjunto fijo de opciones (0, 20, 40, 60, 80, 100 %) en lugar de cualquier valor continuo. *Como el volumen con íconos de 1 a 4 barras en un teléfono.*

## Paso 1 — Verificar el modelo

Ejecutá esta celda **una sola vez** antes de correr cualquier variante. Descarga el modelo si no está disponible localmente.

In [1]:
# Descarga el modelo Hand Landmarker de MediaPipe si no está disponible localmente.
# El modelo (.task) es un bundle TFLite con los pesos preentrenados para detectar
# los 21 puntos clave de la mano.

import urllib.request, os

MODEL_URL  = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
MODEL_PATH = "hand_landmarker.task"

if not os.path.exists(MODEL_PATH):
    print("Descargando modelo...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    tamaño_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)
    print(f"✓ Modelo descargado: {tamaño_mb:.2f} MB")
else:
    tamaño_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)
    print(f"✓ Modelo disponible: {tamaño_mb:.2f} MB")

Descargando modelo...
✓ Modelo descargado: 7.46 MB


## Paso 2 — Funciones compartidas

Esta celda define todas las funciones auxiliares que reutilizan las tres variantes: control de volumen multiplataforma, conversión de coordenadas, dibujo de landmarks y HUD.

In [2]:
# Importa y configura las herramientas compartidas por las tres variantes.
# Ejecutar esta celda es obligatorio antes de correr cualquier variación.

import mediapipe as mp   # Framework de landmarks; proporciona el modelo Hand Landmarker
import cv2               # OpenCV para captura de video y dibujo sobre cuadros
import subprocess        # Para lanzar comandos del SO que controlan el audio en macOS y Linux
import platform          # Detecta el SO en tiempo de ejecución para elegir la API de volumen correcta
import math              # Para la distancia euclidiana en la Variación 1

# --- Alias de la Tasks API de MediaPipe (mejoran la legibilidad) ---
OpcionesBase          = mp.tasks.BaseOptions
DetectorManos         = mp.tasks.vision.HandLandmarker
OpcionesDetectorManos = mp.tasks.vision.HandLandmarkerOptions
ModoEjecucion         = mp.tasks.vision.RunningMode

# --- Control de volumen según el sistema operativo ---
SISTEMA_OPERATIVO = platform.system()  # 'Darwin' = macOS, 'Linux', 'Windows'
print(f"✓ Sistema operativo detectado: {SISTEMA_OPERATIVO}")

VOLUMEN_WINDOWS = None               # Objeto COM de audio; solo se inicializa en Windows
ADVERTENCIA_VOLUMEN_MOSTRADA = False  # Evita repetir el aviso de error en cada frame

if SISTEMA_OPERATIVO == 'Windows':
    try:
        from comtypes import CoInitialize
        from pycaw.pycaw import AudioUtilities
        CoInitialize()  # Inicializa el subsistema COM, requerido por pycaw
        dispositivos = AudioUtilities.GetSpeakers()
        VOLUMEN_WINDOWS = dispositivos.EndpointVolume  # Interfaz COM para leer y escribir el nivel de volumen
        print("✓ Control de volumen de Windows inicializado.")
    except Exception as e:
        VOLUMEN_WINDOWS = None
        print(f"Aviso: no se pudo inicializar pycaw ({e})")


def ajustar_volumen(nivel: int):
    """Establece el volumen del sistema operativo. nivel: 0-100"""
    global ADVERTENCIA_VOLUMEN_MOSTRADA
    nivel = max(0, min(100, nivel))  # Clampea para evitar errores si el gesto excede el rango
    try:
        if SISTEMA_OPERATIVO == 'Darwin':
            subprocess.run(['osascript', '-e', f'set volume output volume {nivel}'], capture_output=True)
        elif SISTEMA_OPERATIVO == 'Linux':
            subprocess.run(['amixer', '-q', 'sset', 'Master', f'{nivel}%'], capture_output=True)
        elif SISTEMA_OPERATIVO == 'Windows':
            if VOLUMEN_WINDOWS is None:
                return
            VOLUMEN_WINDOWS.SetMute(0, None)  # Desmutea primero; de lo contrario el cambio de nivel no tiene efecto
            VOLUMEN_WINDOWS.SetMasterVolumeLevelScalar(nivel / 100.0, None)  # La API COM espera 0.0–1.0
    except Exception as e:
        if not ADVERTENCIA_VOLUMEN_MOSTRADA:  # Muestra el error una sola vez, no en cada frame del loop
            print(f"Aviso: no se pudo aplicar el volumen ({e})")
            ADVERTENCIA_VOLUMEN_MOSTRADA = True


def y_a_volumen(coordenada_y: float) -> int:
    """Convierte coordenada y normalizada [0,1] a porcentaje [0,100]."""
    # Invierte el eje Y: en imagen y=0 es arriba, pero semánticamente 'arriba' = volumen alto
    return max(0, min(100, int((1.0 - coordenada_y) * 100)))


# --- Constantes de color BGR para dibujo ---
COLOR_LANDMARK = (0, 255, 120)   # verde para puntos clave
COLOR_CONEXION = (255, 255, 255) # blanco para líneas
COLOR_MUNECA   = (0, 120, 255)   # naranja para la muñeca (landmark 0)

# Pares de índices que definen los segmentos de la mano (5 dedos + arco de palma)
CONEXIONES_MANO = [
    (0,1),(1,2),(2,3),(3,4),          # Pulgar
    (0,5),(5,6),(6,7),(7,8),          # Índice
    (0,9),(9,10),(10,11),(11,12),     # Medio
    (0,13),(13,14),(14,15),(15,16),   # Anular
    (0,17),(17,18),(18,19),(19,20),   # Meñique
    (5,9),(9,13),(13,17)              # Arco de la palma
]


def dibujar_landmarks(cuadro, puntos_clave):
    """Dibuja el esqueleto de la mano y retorna las posiciones en píxeles."""
    alto, ancho = cuadro.shape[:2]
    # Convertimos coordenadas normalizadas a píxeles
    puntos = [(int(lm.x * ancho), int(lm.y * alto)) for lm in puntos_clave]
    for a, b in CONEXIONES_MANO:
        cv2.line(cuadro, puntos[a], puntos[b], COLOR_CONEXION, 1, cv2.LINE_AA)
    for i, (px, py) in enumerate(puntos):
        color = COLOR_MUNECA if i == 0 else COLOR_LANDMARK
        radio = 7 if i == 0 else 4
        cv2.circle(cuadro, (px, py), radio, color, -1, cv2.LINE_AA)
    return puntos  # El llamador puede usar las posiciones para dibujar elementos adicionales


def dibujar_hud(cuadro, volumen: int, mano_detectada: bool):
    """Superpone la barra de volumen y el estado de detección sobre el cuadro."""
    alto, ancho = cuadro.shape[:2]
    bx, by   = ancho - 50, 30
    bh, bw   = alto - 60, 30
    # Fondo de la barra
    cv2.rectangle(cuadro, (bx, by), (bx + bw, by + bh), (60, 60, 60), -1)
    # Relleno proporcional al volumen; crece desde abajo
    relleno = int(bh * volumen / 100)
    color_r = (0, 200 + int(55 * volumen / 100), 100)  # Verde más brillante a mayor volumen
    cv2.rectangle(cuadro, (bx, by + bh - relleno), (bx + bw, by + bh), color_r, -1)
    cv2.rectangle(cuadro, (bx, by), (bx + bw, by + bh), (180, 180, 180), 1)
    cv2.putText(cuadro, f"{volumen}%", (bx - 5, by + bh + 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
    cv2.putText(cuadro, "VOL", (bx + 2, by - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1, cv2.LINE_AA)
    # Estado de detección
    estado = "MANO DETECTADA" if mano_detectada else "Sin detección"
    color_e = (0, 255, 120) if mano_detectada else (80, 80, 80)
    cv2.putText(cuadro, estado, (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.65, color_e, 1, cv2.LINE_AA)
    cv2.putText(cuadro, "[Q] para salir", (15, alto - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (120, 120, 120), 1, cv2.LINE_AA)


print("✓ Funciones auxiliares cargadas.")

✓ Sistema operativo detectado: Windows
✓ Control de volumen de Windows inicializado.
✓ Funciones auxiliares cargadas.


---

## ✦ Variación 1 — Pinch to Volume

En lugar de usar la posición vertical de la muñeca, medimos la **distancia entre la punta del pulgar (landmark 4) y la punta del índice (landmark 8)**.

```
  ● 4  ← PULGAR (tip)
   \
    ● 3
     \              ● 8  ← ÍNDICE (tip)
      ● 2          /
       \          ● 7
        ● 1      /
         \      ● 6
          ● 5 — ● 0 (muñeca)
```

- Dedos **juntos** (distancia ≈ 0.02) → volumen **0 %**
- Dedos **separados** (distancia ≈ 0.35) → volumen **100 %**

La distancia se calcula en coordenadas normalizadas, por lo que es **independiente de la resolución de la cámara**.

In [7]:
# =================================================================
# VARIACIÓN 1 — Control de volumen por distancia pulgar–índice
# Gesto "pinch": abrí y cerrá el espacio entre pulgar e índice
# =================================================================

# Rango de distancia normalizada calibrado para el gesto pinch
PINCH_MIN = 0.02   # dedos prácticamente juntos → volumen 0 %
PINCH_MAX = 0.35   # dedos completamente separados → volumen 100 %


def distancia_2d(lm_a, lm_b) -> float:
    """Distancia euclidiana entre dos landmarks en coordenadas normalizadas."""
    return math.sqrt((lm_a.x - lm_b.x) ** 2 + (lm_a.y - lm_b.y) ** 2)


def pinch_a_volumen(lm_pulgar, lm_indice) -> int:
    """Mapea la apertura del gesto pinch a un porcentaje de volumen [0,100]."""
    d = distancia_2d(lm_pulgar, lm_indice)
    d_clamped = max(PINCH_MIN, min(PINCH_MAX, d))  # Limita al rango esperado
    return int((d_clamped - PINCH_MIN) / (PINCH_MAX - PINCH_MIN) * 100)


opciones_v1 = OpcionesDetectorManos(
    base_options=OpcionesBase(model_asset_path='hand_landmarker.task'),
    running_mode=ModoEjecucion.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
)

captura  = cv2.VideoCapture(0)
SUAVIZADO        = 0.15
vol_suavizado    = 50
CADA_N           = 5
contador         = 0
ultimo_vol       = None
ADVERTENCIA_VOLUMEN_MOSTRADA = False

print("✦ VARIACIÓN 1 activa. Abrí y cerrá los dedos para cambiar el volumen.")
print("  Presioná Q en la ventana para cerrar.")

try:
    if not captura.isOpened():
        raise RuntimeError("No se pudo acceder a la cámara.")

    with DetectorManos.create_from_options(opciones_v1) as landmarker:
        while True:
            ok, cuadro = captura.read()
            if not ok:
                break

            cuadro = cv2.flip(cuadro, 1)  # Espejo horizontal para interacción intuitiva
            alto, ancho = cuadro.shape[:2]

            cuadro_rgb = cv2.cvtColor(cuadro, cv2.COLOR_BGR2RGB)  # MediaPipe requiere RGB
            imagen_mp  = mp.Image(image_format=mp.ImageFormat.SRGB, data=cuadro_rgb)
            deteccion  = landmarker.detect(imagen_mp)

            mano_detectada  = len(deteccion.hand_landmarks) > 0
            vol_objetivo    = vol_suavizado

            if mano_detectada:
                puntos = deteccion.hand_landmarks[0]

                vol_objetivo = pinch_a_volumen(puntos[4], puntos[8])

                # Convertimos las puntas de control a píxeles para dibujar la línea de apertura
                px_pulgar = (int(puntos[4].x * ancho), int(puntos[4].y * alto))
                px_indice = (int(puntos[8].x * ancho), int(puntos[8].y * alto))

                dibujar_landmarks(cuadro, puntos)
                cv2.line(cuadro, px_pulgar, px_indice, (0, 200, 255), 2, cv2.LINE_AA)  # Línea de apertura
                cv2.circle(cuadro, px_pulgar, 11, (0, 0, 255), -1, cv2.LINE_AA)   # Rojo = pulgar
                cv2.circle(cuadro, px_indice, 11, (255, 80, 0), -1, cv2.LINE_AA)  # Azul = índice

            # Suavizado exponencial: avanzamos un 15 % de la diferencia restante por frame
            vol_suavizado = vol_suavizado + SUAVIZADO * (vol_objetivo - vol_suavizado)
            vol_mostrado  = int(vol_suavizado)

            contador += 1
            if contador % CADA_N == 0 and vol_mostrado != ultimo_vol:
                ajustar_volumen(vol_mostrado)
                ultimo_vol = vol_mostrado

            dibujar_hud(cuadro, vol_mostrado, mano_detectada)
            cv2.putText(cuadro, "V1: PINCH", (15, 55),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 255), 1, cv2.LINE_AA)
            cv2.putText(cuadro, "Abrí/cerrá pulgar-índice", (15, 73),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.42, (180, 180, 180), 1, cv2.LINE_AA)

            cv2.imshow('Variación 1 — Pinch to Volume', cuadro)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                print("✦ Saliendo...")
                break
finally:
    captura.release()
    cv2.destroyAllWindows()
    print("✓ Sesión cerrada.")

✦ VARIACIÓN 1 activa. Abrí y cerrá los dedos para cambiar el volumen.
  Presioná Q en la ventana para cerrar.
✦ Saliendo...
✓ Sesión cerrada.


---

## ✦ Variación 2 — Dos manos

Usamos **dos manos simultáneamente**, cada una controlando una variable diferente:

- **Mano derecha** → posición vertical de la muñeca → **volumen** (misma lógica que el notebook original)
- **Mano izquierda** → posición vertical de la muñeca → **brillo de la imagen** mostrada

```
  Mano IZQUIERDA          Mano DERECHA
  ──────────────          ────────────
  ↑ arriba = más brillo   ↑ arriba = volumen alto
  ↓ abajo  = menos brillo ↓ abajo  = volumen bajo
```

MediaPipe asigna la etiqueta `handedness` a cada mano. Como la imagen está espejada (flip horizontal), la etiqueta `'Right'` corresponde a la **mano real derecha** del usuario.

El **brillo** se aplica con `cv2.convertScaleAbs(cuadro, alpha=factor)`: `alpha=1.0` no modifica la imagen, `alpha>1.0` la aclara, `alpha<1.0` la oscurece.

In [8]:
# =================================================================
# VARIACIÓN 2 — Dos manos: derecha = volumen, izquierda = brillo
# Requiere estar frente a la cámara con ambas manos visibles.
# =================================================================

opciones_v2 = OpcionesDetectorManos(
    base_options=OpcionesBase(model_asset_path='hand_landmarker.task'),
    running_mode=ModoEjecucion.IMAGE,
    num_hands=2,                       # Detecta ambas manos al mismo tiempo
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
)

captura          = cv2.VideoCapture(0)
SUAVIZADO        = 0.15
vol_suavizado    = 50
brillo_suavizado = 50   # 50 % = factor alpha=1.0 = brillo sin modificar
CADA_N           = 5
contador         = 0
ultimo_vol       = None
ADVERTENCIA_VOLUMEN_MOSTRADA = False

print("✦ VARIACIÓN 2 activa.")
print("  Mano DERECHA: sube/baja para controlar VOLUMEN.")
print("  Mano IZQUIERDA: sube/baja para controlar BRILLO.")
print("  Presioná Q para cerrar.")

try:
    if not captura.isOpened():
        raise RuntimeError("No se pudo acceder a la cámara.")

    with DetectorManos.create_from_options(opciones_v2) as landmarker:
        while True:
            ok, cuadro = captura.read()
            if not ok:
                break

            cuadro = cv2.flip(cuadro, 1)
            alto, ancho = cuadro.shape[:2]

            cuadro_rgb = cv2.cvtColor(cuadro, cv2.COLOR_BGR2RGB)
            imagen_mp  = mp.Image(image_format=mp.ImageFormat.SRGB, data=cuadro_rgb)
            deteccion  = landmarker.detect(imagen_mp)

            vol_objetivo    = vol_suavizado
            brillo_objetivo = brillo_suavizado
            n_manos         = len(deteccion.hand_landmarks)

            # Extraemos los objetivos de cada mano antes de aplicar brillo
            manos_info = []
            for i in range(n_manos):
                puntos      = deteccion.hand_landmarks[i]
                # handedness[i][0].category_name: 'Right' o 'Left'
                # En imagen espejada: 'Right' = mano real derecha del usuario
                lateralidad = deteccion.handedness[i][0].category_name
                muneca_y    = puntos[0].y  # Landmark 0 = muñeca
                if lateralidad == 'Right':
                    vol_objetivo = y_a_volumen(muneca_y)
                else:
                    brillo_objetivo = y_a_volumen(muneca_y)  # Reutilizamos el mismo mapeo Y→porcentaje
                manos_info.append((puntos, lateralidad))

            # Suavizado exponencial de ambas variables
            vol_suavizado    = vol_suavizado    + SUAVIZADO * (vol_objetivo    - vol_suavizado)
            brillo_suavizado = brillo_suavizado + SUAVIZADO * (brillo_objetivo - brillo_suavizado)
            vol_mostrado    = int(vol_suavizado)
            brillo_mostrado = int(brillo_suavizado)

            # Aplicamos brillo al frame crudo antes de dibujar la UI
            # Así la interfaz queda legible independientemente del nivel de brillo
            alpha = brillo_mostrado / 50.0  # 50 % → alpha=1.0 (sin cambio)
            cuadro_v = cv2.convertScaleAbs(cuadro, alpha=alpha, beta=0)

            # Dibujamos landmarks y etiqueta de control para cada mano
            for puntos, lateralidad in manos_info:
                dibujar_landmarks(cuadro_v, puntos)
                px = (int(puntos[0].x * ancho), int(puntos[0].y * alto))
                if lateralidad == 'Right':
                    cv2.putText(cuadro_v, f"VOL: {vol_mostrado}%", (px[0] - 40, px[1] + 28),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 120), 1, cv2.LINE_AA)
                else:
                    cv2.putText(cuadro_v, f"BRILLO: {brillo_mostrado}%", (px[0] - 50, px[1] + 28),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 1, cv2.LINE_AA)

            contador += 1
            if contador % CADA_N == 0 and vol_mostrado != ultimo_vol:
                ajustar_volumen(vol_mostrado)
                ultimo_vol = vol_mostrado

            dibujar_hud(cuadro_v, vol_mostrado, n_manos > 0)
            cv2.putText(cuadro_v, f"BRILLO: {brillo_mostrado}%", (15, 55),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1, cv2.LINE_AA)
            cv2.putText(cuadro_v, "V2: DOS MANOS", (15, 73),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.42, (180, 180, 180), 1, cv2.LINE_AA)

            cv2.imshow('Variación 2 — Dos Manos', cuadro_v)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                print("✦ Saliendo...")
                break
finally:
    captura.release()
    cv2.destroyAllWindows()
    print("✓ Sesión cerrada.")

✦ VARIACIÓN 2 activa.
  Mano DERECHA: sube/baja para controlar VOLUMEN.
  Mano IZQUIERDA: sube/baja para controlar BRILLO.
  Presioná Q para cerrar.
✦ Saliendo...
✓ Sesión cerrada.


---

## ✦ Variación 3 — Conteo de Dedos

Detectamos cuántos dedos están **extendidos** (0 a 5) y mapeamos ese conteo a 6 niveles discretos de volumen.

**¿Cómo sabemos si un dedo está extendido?**

Comparamos la coordenada Y de la **punta** (*tip*) con la de la **articulación media** (*PIP*):

```
  ● tip (8)   ← y pequeño (más arriba) → dedo EXTENDIDO
  |
  ● DIP (7)
  |
  ● PIP (6)   ← referencia de comparación
  |
  ● MCP (5)
  |
[palma — 0]
```

El **pulgar** es un caso especial: su movimiento es **lateral** (eje X), no vertical. Comparamos la punta (landmark 4) con la articulación IP (landmark 3) en el eje X.

| Dedos levantados | Volumen |
|:---:|:---:|
| 0 ✊ | 0 % |
| 1 ☝️ | 20 % |
| 2 ✌️ | 40 % |
| 3 | 60 % |
| 4 | 80 % |
| 5 🖐️ | 100 % |

In [10]:
# =================================================================
# VARIACIÓN 3 — Función auxiliar: contar dedos extendidos
# =================================================================

# Pares (índice_tip, índice_pip) para los dedos 2-5
# El pulgar (dedo 1) se trata aparte por su orientación lateral
PARES_DEDOS = [
    (8,  6),   # Índice:  tip vs PIP
    (12, 10),  # Medio:   tip vs PIP
    (16, 14),  # Anular:  tip vs PIP
    (20, 18),  # Meñique: tip vs PIP
]

# Índices de las puntas de los 5 dedos (para el feedback visual)
TIPS = [4, 8, 12, 16, 20]

# Mapeo de cantidad de dedos extendidos a porcentaje de volumen
NIVELES = {0: 0, 1: 20, 2: 40, 3: 60, 4: 80, 5: 100}


def contar_dedos(puntos_clave) -> int:
    """
    Cuenta cuántos dedos están extendidos (0-5).
    Calibrado para la mano derecha en imagen espejada (flip horizontal).
    """
    count = 0
    # Pulgar: en imagen espejada con mano derecha, extendido → tip.x > ip.x
    if puntos_clave[4].x > puntos_clave[3].x:
        count += 1
    # Dedos 2-5: extendido = punta está más arriba que PIP (y menor en imagen)
    for tip_i, pip_i in PARES_DEDOS:
        if puntos_clave[tip_i].y < puntos_clave[pip_i].y:
            count += 1
    return count


def estado_dedos(puntos_clave) -> list:
    """Retorna una lista de 5 booleanos: True = dedo extendido, para cada dedo (pulgar al meñique)."""
    estados = [puntos_clave[4].x > puntos_clave[3].x]  # Pulgar
    estados += [puntos_clave[t].y < puntos_clave[p].y for t, p in PARES_DEDOS]
    return estados


print("✓ Funciones contar_dedos y estado_dedos cargadas.")

✓ Funciones contar_dedos y estado_dedos cargadas.


In [11]:
# =================================================================
# VARIACIÓN 3 — Loop: conteo de dedos → volumen discreto
# =================================================================

opciones_v3 = OpcionesDetectorManos(
    base_options=OpcionesBase(model_asset_path='hand_landmarker.task'),
    running_mode=ModoEjecucion.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
)

captura      = cv2.VideoCapture(0)
CADA_N       = 5
contador     = 0
ultimo_count = -1   # Evita re-aplicar el mismo nivel si el conteo no cambió
ADVERTENCIA_VOLUMEN_MOSTRADA = False

print("✦ VARIACIÓN 3 activa. Levantá dedos para subir el volumen.")
print("  0 dedos = 0 % | 1 = 20 % | 2 = 40 % | 3 = 60 % | 4 = 80 % | 5 = 100 %")
print("  Presioná Q para cerrar.")

try:
    if not captura.isOpened():
        raise RuntimeError("No se pudo acceder a la cámara.")

    with DetectorManos.create_from_options(opciones_v3) as landmarker:
        while True:
            ok, cuadro = captura.read()
            if not ok:
                break

            cuadro = cv2.flip(cuadro, 1)
            alto, ancho = cuadro.shape[:2]

            cuadro_rgb = cv2.cvtColor(cuadro, cv2.COLOR_BGR2RGB)
            imagen_mp  = mp.Image(image_format=mp.ImageFormat.SRGB, data=cuadro_rgb)
            deteccion  = landmarker.detect(imagen_mp)

            mano_detectada = len(deteccion.hand_landmarks) > 0
            count   = 0
            volumen = 0

            if mano_detectada:
                puntos  = deteccion.hand_landmarks[0]
                count   = contar_dedos(puntos)
                volumen = NIVELES[count]
                estados = estado_dedos(puntos)

                lista_px = dibujar_landmarks(cuadro, puntos)

                # Feedback visual: verde en puntas extendidas, rojo en plegadas
                for j, tip_idx in enumerate(TIPS):
                    px_tip = lista_px[tip_idx]
                    color  = (0, 220, 0) if estados[j] else (0, 0, 220)
                    cv2.circle(cuadro, px_tip, 13, color, -1, cv2.LINE_AA)
                    cv2.circle(cuadro, px_tip, 13, (255, 255, 255), 1, cv2.LINE_AA)  # Borde blanco

                # Número grande en el centro que muestra el conteo actual
                cx, cy = ancho // 2, alto // 2
                cv2.putText(cuadro, str(count), (cx - 25, cy + 25),
                            cv2.FONT_HERSHEY_SIMPLEX, 3.5, (255, 255, 255), 8, cv2.LINE_AA)
                cv2.putText(cuadro, str(count), (cx - 25, cy + 25),
                            cv2.FONT_HERSHEY_SIMPLEX, 3.5, (0, 200, 255), 4, cv2.LINE_AA)

            # El nivel discreto se aplica solo cuando el conteo cambia (no necesita suavizado)
            contador += 1
            if contador % CADA_N == 0 and count != ultimo_count:
                ajustar_volumen(volumen)
                ultimo_count = count

            dibujar_hud(cuadro, volumen, mano_detectada)
            cv2.putText(cuadro, "V3: CONTEO DE DEDOS", (15, 55),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 200, 255), 1, cv2.LINE_AA)

            # Escala de niveles en la esquina inferior izquierda: resalta el nivel activo
            for d in range(6):
                color_nivel = (0, 220, 0) if d == count else (80, 80, 80)
                cv2.putText(cuadro, f"{d} dedo{'s' if d != 1 else ''}: {NIVELES[d]}%",
                            (15, alto - 110 + d * 16),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.38, color_nivel, 1, cv2.LINE_AA)

            cv2.imshow('Variación 3 — Conteo de Dedos', cuadro)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                print("✦ Saliendo...")
                break
finally:
    captura.release()
    cv2.destroyAllWindows()
    print("✓ Sesión cerrada.")

✦ VARIACIÓN 3 activa. Levantá dedos para subir el volumen.
  0 dedos = 0 % | 1 = 20 % | 2 = 40 % | 3 = 60 % | 4 = 80 % | 5 = 100 %
  Presioná Q para cerrar.
✦ Saliendo...
✓ Sesión cerrada.


---

## Conclusiones

Las tres variantes implementan la misma estructura base (loop de cámara + detección + mapeo + HUD), pero difieren en la **señal de entrada** que extraen de los landmarks:

| Variante | Señal extraída | Tipo de control |
|---|---|---|
| V1 — Pinch | Distancia euclidiana entre 2 landmarks | Continuo, suave |
| V2 — Dos manos | Posición Y de la muñeca (×2) | Continuo, dos variables |
| V3 — Conteo | Estado booleano de cada dedo | Discreto, 6 niveles |

### Para seguir explorando

- **Combinar V1 y V3**: usar el conteo de dedos para seleccionar el modo de control (pinch vs posición).
- **Agregar una barra de brillo** en la V2 similar a la barra de volumen del HUD.
- **Calibración dinámica**: que el usuario defina los valores mínimo y máximo del pinch con un gesto inicial.